# create the off labels; 3 sec before, after and off animations

In [5]:
import numpy as np
import pandas as pd

In [6]:
tasks_all = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/all_tasks.csv")
fixations_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fixations_all_rows.csv")
gaze_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_all_rows.csv")

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_88621/3396532321.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  fixations_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fixations_all_rows.csv")
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_88621/3396532321.py:3: DtypeWarning: Columns (16,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  gaze_df = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_all_rows.csv")


In [7]:
import numpy as np
import pandas as pd

def assign_on_off_labels(
    samples_df: pd.DataFrame,
    tasks_all: pd.DataFrame,
    *,
    pre_seconds,
    post_seconds,
    decode_task=None,                    # NEW (optional) decode_task(tid) -> {"code","type",...}
    no_anim_ids: set[str] | None = None, # explicit non-animated ids
    treat_no_anim_as_on: bool = False,   # keep as you had
    time_col: str = "TIME",
    label_col: str = "label",
) -> pd.DataFrame:
    """
    Labels rows as 'ON'/'OFF' (unchanged from your logic) and ALSO adds:
      - block_id   : groups ON + OFF windows for the same task
      - phase      : 'OFF_PRE', 'ON', 'OFF_POST'
      - block_anim : bool (animated task?)

    Overlap policy for OFF windows: keep the OFF phase closest to its task boundary.
    'ON' phase is assigned last and wins inside [t_start, t_end].
    """
    def _is_animated(tid: str) -> bool:
        tid = str(tid)
        if no_anim_ids is not None and tid in no_anim_ids:
            return False
        if decode_task is not None:
            try:
                info = decode_task(tid) or {}
                code = (info.get("code") or "").lower()
                if "null" in code:
                    return False
            except Exception:
                pass
        return True

    def _anim_type(tid: str) -> str:
        if decode_task is None:
            return "Unknown"
        try:
            info = decode_task(str(tid)) or {}
            return info.get("type", "Unknown")
        except Exception:
            return "Unknown"

    df = samples_df.copy(deep=True)
    if label_col not in df.columns:
        df[label_col] = "UNLABELED"
    df[time_col] = pd.to_numeric(df[time_col], errors="coerce")

    # normalize tasks table
    t = tasks_all.rename(columns={
        "Video Name": "Video Name (folder)",
        "User ID": "User ID (file)",
        "Starting time in seconds": "t_start",
        "Finishing time in seconds": "t_end",
    }).copy()
    t["Task Id"] = t["Task Id"].astype(str)
    t["t_start"] = pd.to_numeric(t["t_start"], errors="coerce")
    t["t_end"]   = pd.to_numeric(t["t_end"],   errors="coerce")
    t = t.dropna(subset=["t_start","t_end"])
    t = t[t["t_end"] >= t["t_start"]].copy()

    # per-task metadata used for block grouping
    t["block_anim"] = t["Task Id"].apply(_is_animated)
    t["block_id"] = (
        t["Video Name (folder)"].astype(str) + "|" +
        t["User ID (file)"].astype(str) + "|" +
        t["Task Id"].astype(str)
    )

    # ensure new columns exist (we won't touch your label values)
    for col in ["phase", "block_id", "block_anim"]:
        if col not in df.columns:
            df[col] = pd.NA

    # temp distance column to resolve OFF overlaps
    df["_phase_dist"] = np.inf

    gkeys = ["Video Name (folder)", "User ID (file)"]

    # ---- 1) Assign OFF pre/post (for ALL tasks) LOWER PRIORITY ----
    for (vn, uid), grp in t.groupby(gkeys):
        sel = (df["Video Name (folder)"] == vn) & (df["User ID (file)"] == uid)
        if not sel.any():
            continue
        for _, r in grp.iterrows():
            s, e = float(r["t_start"]), float(r["t_end"])
            pre_s  = max(0.0, s - float(pre_seconds))
            post_e = e + float(post_seconds)

            # OFF_PRE: [pre_s, s)
            m_pre = sel & df[time_col].between(pre_s, s, inclusive="left")
            if m_pre.any():
                dist = (s - df.loc[m_pre, time_col]).abs()
                keep = dist < df.loc[m_pre, "_phase_dist"]
                idx = df.loc[m_pre].index[keep]
                df.loc[idx, ["phase","block_id","block_anim"]] = (
                    "OFF_PRE", r["block_id"], r["block_anim"]
                )
                df.loc[idx, "_phase_dist"] = dist.loc[keep]
                # your original OFF labeling rule (unchanged)
                df.loc[idx, label_col] = np.where(df.loc[idx, label_col] == "ON", df.loc[idx, label_col], "OFF")

            # OFF_POST: (e, post_e]
            m_post = sel & df[time_col].between(e, post_e, inclusive="right")
            if m_post.any():
                dist = (df.loc[m_post, time_col] - e).abs()
                keep = dist < df.loc[m_post, "_phase_dist"]
                idx = df.loc[m_post].index[keep]
                df.loc[idx, ["phase","block_id","block_anim"]] = (
                    "OFF_POST", r["block_id"], r["block_anim"]
                )
                df.loc[idx, "_phase_dist"] = dist.loc[keep]
                # your original OFF labeling rule (unchanged)
                df.loc[idx, label_col] = np.where(df.loc[idx, label_col] == "ON", df.loc[idx, label_col], "OFF")

    t_noanim = t[~t["block_anim"]].copy()
    if not treat_no_anim_as_on and not t_noanim.empty:
        for (vn, uid), grp in t_noanim.groupby(gkeys):
            sel = (df["Video Name (folder)"] == vn) & (df["User ID (file)"] == uid)
            if not sel.any():
                continue
            for _, r in grp.iterrows():
                s, e = float(r["t_start"]), float(r["t_end"])
                m_no = sel & df[time_col].between(s, e, inclusive="both")
                if m_no.any():
                    # label OFF (your rule), but set phase ON for grouping later
                    df.loc[m_no, label_col] = np.where(df.loc[m_no, label_col] == "ON", df.loc[m_no, label_col], "OFF")
                    df.loc[m_no, ["phase","block_id","block_anim"]] = (
                        "ON", r["block_id"], False
                    )
                    df.loc[m_no, "_phase_dist"] = 0.0

    # ---- 3) Animated task ON (HIGHER PRIORITY) ----
    t_anim = t[t["block_anim"]].copy()
    if not t_anim.empty:
        for (vn, uid), grp in t_anim.groupby(gkeys):
            sel = (df["Video Name (folder)"] == vn) & (df["User ID (file)"] == uid)
            if not sel.any():
                continue
            for _, r in grp.iterrows():
                s, e = float(r["t_start"]), float(r["t_end"])
                m_on = sel & df[time_col].between(s, e, inclusive="both")
                if m_on.any():
                    # your original ON labeling
                    df.loc[m_on, label_col] = "ON"
                    # set block grouping info + phase
                    df.loc[m_on, ["phase","block_id","block_anim"]] = (
                        "ON", r["block_id"], True
                    )
                    df.loc[m_on, "_phase_dist"] = 0.0

    # ---- 4) Optional: treat no-anim as ON if you want later ----
    if treat_no_anim_as_on and not t_noanim.empty:
        for (vn, uid), grp in t_noanim.groupby(gkeys):
            sel = (df["Video Name (folder)"] == vn) & (df["User ID (file)"] == uid)
            if not sel.any():
                continue
            for _, r in grp.iterrows():
                s, e = float(r["t_start"]), float(r["t_end"])
                m_on = sel & df[time_col].between(s, e, inclusive="both")
                if m_on.any():
                    df.loc[m_on, label_col] = "ON"
                    df.loc[m_on, ["phase","block_id","block_anim"]] = (
                        "ON", r["block_id"], False
                    )
                    df.loc[m_on, "_phase_dist"] = 0.0

    df.drop(columns=["_phase_dist"], inplace=True, errors="ignore")
    return df

In [8]:
# 1) Run your usual labeling (now augmented with block info)
fix_labeled = assign_on_off_labels(
    samples_df=fixations_df,     # or your merged samples
    tasks_all=tasks_all,
    pre_seconds=4.0,
    post_seconds=4.0,
    no_anim_ids={"1.4","2.2","3.9"},  # optional
    treat_no_anim_as_on=False    # keep labels as-is
)
fix_labeled.to_csv("outputs/fix_labeled_with_blocks.csv", index=False)

gaze_labeled = assign_on_off_labels(
    samples_df=gaze_df,     # or your merged samples
    tasks_all=tasks_all,
    pre_seconds=4.0,
    post_seconds=4.0,
    no_anim_ids={"1.4","2.2","3.9"},  # optional
    treat_no_anim_as_on=False    # keep labels as-is
)
gaze_labeled.to_csv("outputs/gaze_labeled_with_blocks.csv", index=False)



# 2) Later, compute deltas PER BLOCK using 'phase' (not 'label'):
#    Δ = mean(phase=='ON') - mean(phase=='OFF_PRE') within the same block_id